In [1]:
from config import TrainingConfig

config = TrainingConfig()
config.validate()

print(config)
print("Configuration validation passed.")

TrainingConfig(epochs=15, max_memory=1000, batch_size=50, discount_factor=0.95, epsilon_start=1.0, epsilon_min=0.05, epsilon_decay=0.995, target_update_frequency=10, random_seed=42)
Configuration validation passed.


In [2]:
bad_config = TrainingConfig(epsilon_start=1.5)

try:
    bad_config.validate()
except ValueError as error:
    print(f"Validation correctly rejected invalid configuration: {error}")

Validation correctly rejected invalid configuration: epsilon_start must be between 0.0 and 1.0.


In [3]:
import numpy as np

from environment.treasure_maze import Action, TreasureMaze


test_maze = np.array([
    [1, 1, 0, 0],
    [0, 1, 1, 0],
    [0, 0, 1, 1],
    [0, 0, 0, 1],
])

environment = TreasureMaze(test_maze)

print("Starting state:", environment.state)
print("Target:", environment.target)
print("Valid actions:", environment.valid_actions())
print("Observation shape:", environment.observe().shape)
print("Game status:", environment.game_status())

Starting state: (0, 0, 'start')
Target: (3, 3)
Valid actions: [<Action.RIGHT: 2>]
Observation shape: (1, 16)
Game status: not_over


In [4]:
# Reset the environment before testing movement.
environment.reset((0, 0))

print("=== VALID MOVEMENT TEST ===")

state, reward, status = environment.act(Action.RIGHT)

print("State after RIGHT:", environment.state)
print("Reward:", reward)
print("Total reward:", environment.total_reward)
print("Status:", status)
print("Valid actions now:", environment.valid_actions())


print("\n=== INVALID MOVEMENT TEST ===")

# UP is invalid from position (0, 1) because it would leave the maze.
state, reward, status = environment.act(Action.UP)

print("State after invalid UP:", environment.state)
print("Reward:", reward)
print("Total reward:", environment.total_reward)
print("Status:", status)

=== VALID MOVEMENT TEST ===
State after RIGHT: (0, 1, 'valid')
Reward: -0.04
Total reward: -0.04
Status: not_over
Valid actions now: [<Action.LEFT: 0>, <Action.DOWN: 3>]

=== INVALID MOVEMENT TEST ===
State after invalid UP: (0, 1, 'invalid')
Reward: -0.75
Total reward: -0.79
Status: not_over


In [5]:
print("=== MAZE VALIDATION TESTS ===")

# Test 1: Invalid maze cell value
invalid_maze = np.array([
    [1, 1],
    [0, 2],
])

try:
    TreasureMaze(invalid_maze)
except ValueError as error:
    print("PASS - Invalid cell value rejected:")
    print(error)


# Test 2: Blocked starting position
blocked_start_maze = np.array([
    [1, 0],
    [1, 1],
])

try:
    TreasureMaze(blocked_start_maze, pirate=(0, 1))
except ValueError as error:
    print("\nPASS - Blocked starting position rejected:")
    print(error)


# Test 3: Out-of-bounds starting position
valid_maze = np.array([
    [1, 1],
    [1, 1],
])

try:
    TreasureMaze(valid_maze, pirate=(5, 5))
except ValueError as error:
    print("\nPASS - Out-of-bounds position rejected:")
    print(error)


# Test 4: Unsupported action
environment.reset((0, 0))

try:
    environment.act(99)
except ValueError as error:
    print("\nPASS - Unsupported action rejected:")
    print(error)

=== MAZE VALIDATION TESTS ===
PASS - Invalid cell value rejected:
maze cells must contain only 0 for blocked cells or 1 for open cells.

PASS - Blocked starting position rejected:
pirate position must be located on an open maze cell.

PASS - Out-of-bounds position rejected:
pirate position must be within the maze boundaries.

PASS - Unsupported action rejected:
Unsupported action: 99


In [6]:
from agent.replay_memory import Experience, ReplayMemory


print("=== REPLAY MEMORY TESTS ===")

memory = ReplayMemory(
    capacity=3,
    random_seed=42,
)

# Create four simple experiences.
for index in range(4):
    experience = Experience(
        state=np.array([[index]], dtype=float),
        action=index % 4,
        reward=float(index),
        next_state=np.array([[index + 1]], dtype=float),
        status="not_over",
    )

    memory.add(experience)

print("Memory size:", len(memory))
print("Capacity:", memory.capacity)

# Because capacity is 3, adding four experiences should
# automatically discard the oldest experience.
print("\nStored experiences:")
for experience in memory._memory:
    print(
        "State:",
        experience.state.flatten()[0],
        "| Action:",
        experience.action,
        "| Reward:",
        experience.reward,
    )

sample = memory.sample(2)

print("\nSample size:", len(sample))

for experience in sample:
    print(
        "Sampled state:",
        experience.state.flatten()[0],
        "| Reward:",
        experience.reward,
    )


print("\n=== REPLAY MEMORY VALIDATION ===")

try:
    ReplayMemory(capacity=0)
except ValueError as error:
    print("PASS - Invalid capacity rejected:")
    print(error)

try:
    memory.sample(0)
except ValueError as error:
    print("\nPASS - Invalid batch size rejected:")
    print(error)

empty_memory = ReplayMemory()

try:
    empty_memory.sample(1)
except ValueError as error:
    print("\nPASS - Empty memory sampling rejected:")
    print(error)

=== REPLAY MEMORY TESTS ===
Memory size: 3
Capacity: 3

Stored experiences:
State: 1.0 | Action: 1 | Reward: 1.0
State: 2.0 | Action: 2 | Reward: 2.0
State: 3.0 | Action: 3 | Reward: 3.0

Sample size: 2
Sampled state: 1.0 | Reward: 1.0
Sampled state: 3.0 | Reward: 3.0

=== REPLAY MEMORY VALIDATION ===
PASS - Invalid capacity rejected:
capacity must be greater than zero.

PASS - Invalid batch size rejected:
batch_size must be greater than zero.

PASS - Empty memory sampling rejected:
cannot sample from empty replay memory.


In [7]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles and folders Jupyter can see:")
print(os.listdir())

Current working directory:
C:\Users\mbile\Documents\CS499_PirateIntelligentAgent\PirateIntelligentAgent_Enhanced

Files and folders Jupyter can see:
['.ipynb_checkpoints', 'agent', 'config.py', 'environment', 'test_enhancement.ipynb', 'training', '__pycache__']


In [8]:
import os

print(os.listdir())

print("\nAgent folder:")
print(os.listdir("agent"))

['.ipynb_checkpoints', 'agent', 'config.py', 'environment', 'test_enhancement.ipynb', 'training', '__pycache__']

Agent folder:
['.ipynb_checkpoints', 'pirate_agent.py', 'replay_memory.py', '__init__.py', '__pycache__']


In [9]:
from agent.replay_memory import Experience, ReplayMemory


print("=== REPLAY MEMORY TESTS ===")

memory = ReplayMemory(
    capacity=3,
    random_seed=42,
)

# Create four simple experiences.
for index in range(4):
    experience = Experience(
        state=np.array([[index]], dtype=float),
        action=index % 4,
        reward=float(index),
        next_state=np.array([[index + 1]], dtype=float),
        status="not_over",
    )

    memory.add(experience)

print("Memory size:", len(memory))
print("Capacity:", memory.capacity)

# Because capacity is 3, adding four experiences should
# automatically discard the oldest experience.
print("\nStored experiences:")
for experience in memory._memory:
    print(
        "State:",
        experience.state.flatten()[0],
        "| Action:",
        experience.action,
        "| Reward:",
        experience.reward,
    )

sample = memory.sample(2)

print("\nSample size:", len(sample))

for experience in sample:
    print(
        "Sampled state:",
        experience.state.flatten()[0],
        "| Reward:",
        experience.reward,
    )


print("\n=== REPLAY MEMORY VALIDATION ===")

try:
    ReplayMemory(capacity=0)
except ValueError as error:
    print("PASS - Invalid capacity rejected:")
    print(error)

try:
    memory.sample(0)
except ValueError as error:
    print("\nPASS - Invalid batch size rejected:")
    print(error)

empty_memory = ReplayMemory()

try:
    empty_memory.sample(1)
except ValueError as error:
    print("\nPASS - Empty memory sampling rejected:")
    print(error)

=== REPLAY MEMORY TESTS ===
Memory size: 3
Capacity: 3

Stored experiences:
State: 1.0 | Action: 1 | Reward: 1.0
State: 2.0 | Action: 2 | Reward: 2.0
State: 3.0 | Action: 3 | Reward: 3.0

Sample size: 2
Sampled state: 1.0 | Reward: 1.0
Sampled state: 3.0 | Reward: 3.0

=== REPLAY MEMORY VALIDATION ===
PASS - Invalid capacity rejected:
capacity must be greater than zero.

PASS - Invalid batch size rejected:
batch_size must be greater than zero.

PASS - Empty memory sampling rejected:
cannot sample from empty replay memory.


In [10]:
from agent.pirate_agent import PirateAgent


print("=== PIRATE AGENT TESTS ===")

# The 4x4 test maze contains 16 state values.
agent = PirateAgent(
    state_size=16,
    action_size=4,
    random_seed=42,
)

print("Agent created successfully.")
print("State size:", agent.state_size)
print("Action size:", agent.action_size)


# Test neural-network prediction.
environment.reset((0, 0))
state = environment.observe()

q_values = agent.predict(state)

print("\nPrediction shape:", q_values.shape)
print("Q-values:", q_values)


# Test fully exploratory action selection.
random_generator = np.random.default_rng(42)

valid_actions = environment.valid_actions()

exploratory_action = agent.select_action(
    state=state,
    valid_actions=valid_actions,
    epsilon=1.0,
    random_generator=random_generator,
)

print("\nValid actions:", valid_actions)
print("Exploratory action:", exploratory_action)
print(
    "Exploratory action is valid:",
    exploratory_action in valid_actions,
)


# Test fully exploitative action selection.
exploitative_action = agent.select_action(
    state=state,
    valid_actions=valid_actions,
    epsilon=0.0,
    random_generator=random_generator,
)

print("\nExploitative action:", exploitative_action)
print(
    "Exploitative action is valid:",
    exploitative_action in valid_actions,
)


print("\n=== PIRATE AGENT VALIDATION ===")

try:
    PirateAgent(state_size=0)
except ValueError as error:
    print("PASS - Invalid state size rejected:")
    print(error)


try:
    agent.predict(
        np.array([[1.0, 2.0, 3.0]])
    )
except ValueError as error:
    print("\nPASS - Invalid state rejected:")
    print(error)


try:
    agent.select_action(
        state=state,
        valid_actions=[],
        epsilon=0.5,
        random_generator=random_generator,
    )
except ValueError as error:
    print("\nPASS - Empty action list rejected:")
    print(error)


try:
    agent.select_action(
        state=state,
        valid_actions=valid_actions,
        epsilon=1.5,
        random_generator=random_generator,
    )
except ValueError as error:
    print("\nPASS - Invalid epsilon rejected:")
    print(error)


# Verify target-network synchronization.
agent.update_target_network()

target_q_values = agent.predict(
    state,
    use_target=True,
)

print(
    "\nTarget network synchronized:",
    np.allclose(q_values, target_q_values),
)

=== PIRATE AGENT TESTS ===
Agent created successfully.
State size: 16
Action size: 4

Prediction shape: (1, 4)
Q-values: [[ 0.47063896 -0.18786573  0.06801315 -0.0231724 ]]

Valid actions: [<Action.RIGHT: 2>]
Exploratory action: 2
Exploratory action is valid: True

Exploitative action: 2
Exploitative action is valid: True

=== PIRATE AGENT VALIDATION ===
PASS - Invalid state size rejected:
state_size must be greater than zero.

PASS - Invalid state rejected:
state must contain 16 values.

PASS - Empty action list rejected:
valid_actions cannot be empty.

PASS - Invalid epsilon rejected:
epsilon must be between 0.0 and 1.0.

Target network synchronized: True


In [11]:
from training.trainer import Trainer


print("=== TRAINER INTEGRATION TEST ===")

# Use the existing 4x4 test maze.
training_environment = TreasureMaze(test_maze)

training_agent = PirateAgent(
    state_size=test_maze.size,
    action_size=4,
    random_seed=42,
)

# Keep this test intentionally small and fast.
test_config = TrainingConfig(
    epochs=2,
    max_memory=100,
    batch_size=10,
    discount_factor=0.95,
    epsilon_start=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.995,
    target_update_frequency=1,
    random_seed=42,
)

trainer = Trainer(
    environment=training_environment,
    agent=training_agent,
    config=test_config,
)

print("Trainer created successfully.")
print("Initial epsilon:", trainer.epsilon)
print("Replay memory size:", len(trainer.memory))

metrics = trainer.train()

print("\n=== TRAINING RESULTS ===")
print("Epochs completed:", metrics.epochs_completed)
print("Wins:", metrics.wins)
print("Losses:", metrics.losses)
print("Win rate:", metrics.win_rate)
print("Final epsilon:", metrics.final_epsilon)
print("Average loss:", metrics.average_loss)
print("Replay memory size:", len(trainer.memory))

=== TRAINER INTEGRATION TEST ===
Trainer created successfully.
Initial epsilon: 1.0
Replay memory size: 0

=== TRAINING RESULTS ===
Epochs completed: 2
Wins: 2
Losses: 0
Win rate: 1.0
Final epsilon: 0.990025
Average loss: 0.005878821601072559
Replay memory size: 48


In [1]:
import numpy as np

from algorithms.astar import AStarPathfinder


print("=== A* PATHFINDING TEST ===")

test_maze = np.array([
    [1, 1, 0, 0],
    [0, 1, 1, 0],
    [0, 0, 1, 1],
    [0, 0, 0, 1],
])

pathfinder = AStarPathfinder(test_maze)

result = pathfinder.search(
    start=(0, 0),
    goal=(3, 3),
)

print("Path found:", result.found)
print("Path:", result.path)
print("Path length:", result.path_length)
print("Nodes explored:", result.nodes_explored)
print("Total cost:", result.total_cost)

=== A* PATHFINDING TEST ===
Path found: True
Path: [(0, 0), (0, 1), (1, 1), (1, 2), (2, 2), (2, 3), (3, 3)]
Path length: 6
Nodes explored: 7
Total cost: 6.0


In [2]:
print("\n=== A* VALIDATION TESTS ===")


# Test 1: Starting position is blocked
try:
    pathfinder.search(
        start=(1, 0),
        goal=(3, 3),
    )
except ValueError as error:
    print("\nPASS - Blocked starting position rejected:")
    print(error)


# Test 2: Goal position is blocked
try:
    pathfinder.search(
        start=(0, 0),
        goal=(0, 2),
    )
except ValueError as error:
    print("\nPASS - Blocked goal position rejected:")
    print(error)


# Test 3: Starting position is outside the maze
try:
    pathfinder.search(
        start=(-1, 0),
        goal=(3, 3),
    )
except ValueError as error:
    print("\nPASS - Out-of-bounds starting position rejected:")
    print(error)


# Test 4: Goal position is outside the maze
try:
    pathfinder.search(
        start=(0, 0),
        goal=(10, 10),
    )
except ValueError as error:
    print("\nPASS - Out-of-bounds goal position rejected:")
    print(error)


=== A* VALIDATION TESTS ===

PASS - Blocked starting position rejected:
start must be located on an open maze cell.

PASS - Blocked goal position rejected:
goal must be located on an open maze cell.

PASS - Out-of-bounds starting position rejected:
start must be within the maze boundaries.

PASS - Out-of-bounds goal position rejected:
goal must be within the maze boundaries.


In [3]:
print("\n=== A* UNREACHABLE GOAL TEST ===")

unreachable_maze = np.array([
    [1, 1, 0, 0],
    [0, 1, 0, 1],
    [0, 0, 0, 1],
    [0, 0, 0, 1],
])

unreachable_pathfinder = AStarPathfinder(
    unreachable_maze
)

result = unreachable_pathfinder.search(
    start=(0, 0),
    goal=(3, 3),
)

print("Path found:", result.found)
print("Path:", result.path)
print("Path length:", result.path_length)
print("Nodes explored:", result.nodes_explored)
print("Total cost:", result.total_cost)


=== A* UNREACHABLE GOAL TEST ===
Path found: False
Path: []
Path length: 0
Nodes explored: 3
Total cost: inf


In [4]:
from algorithms.bfs import BFSPathfinder


print("=== A* VS. BFS COMPARISON ===")

comparison_maze = np.array([
    [1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 0, 0, 0, 0, 0, 1],
    [1, 1, 1, 1, 1, 1, 0, 1],
    [0, 0, 0, 0, 0, 1, 0, 1],
    [1, 1, 1, 1, 0, 1, 1, 1],
    [1, 0, 0, 1, 0, 0, 0, 1],
    [1, 1, 1, 1, 1, 1, 1, 1],
    [0, 0, 0, 0, 0, 0, 0, 1],
])

start = (0, 0)
goal = (7, 7)

astar_pathfinder = AStarPathfinder(
    comparison_maze
)

bfs_pathfinder = BFSPathfinder(
    comparison_maze
)

astar_result = astar_pathfinder.search(
    start=start,
    goal=goal,
)

bfs_result = bfs_pathfinder.search(
    start=start,
    goal=goal,
)


print("\nA* RESULTS")
print("Path found:", astar_result.found)
print("Path length:", astar_result.path_length)
print("Nodes explored:", astar_result.nodes_explored)
print("Total cost:", astar_result.total_cost)


print("\nBFS RESULTS")
print("Path found:", bfs_result.found)
print("Path length:", bfs_result.path_length)
print("Nodes explored:", bfs_result.nodes_explored)
print("Total cost:", bfs_result.total_cost)


print("\nCOMPARISON")

if astar_result.path_length == bfs_result.path_length:
    print(
        "PASS - Both algorithms found "
        "a shortest path of equal length."
    )
else:
    print(
        "WARNING - Path lengths are different."
    )

difference = (
    bfs_result.nodes_explored
    - astar_result.nodes_explored
)

print(
    "Difference in nodes explored:",
    difference,
)

if astar_result.nodes_explored < bfs_result.nodes_explored:
    print(
        "A* explored fewer nodes than BFS."
    )
elif astar_result.nodes_explored == bfs_result.nodes_explored:
    print(
        "A* and BFS explored the same number of nodes."
    )
else:
    print(
        "BFS explored fewer nodes than A* "
        "for this maze."
    )

=== A* VS. BFS COMPARISON ===

A* RESULTS
Path found: True
Path length: 14
Nodes explored: 25
Total cost: 14.0

BFS RESULTS
Path found: True
Path length: 14
Nodes explored: 26
Total cost: 14.0

COMPARISON
PASS - Both algorithms found a shortest path of equal length.
Difference in nodes explored: 1
A* explored fewer nodes than BFS.
